# Stage 04B — supervised DeBERTa trial

One fixed three-epoch recipe, exact saved five prompt folds, offline T4 execution. This notebook is development evaluation and creates no competition submission. Keep private. Protocol clarification commit: 5563d45ef49311afa7ca0cacc184bb2ed46f3ba8.

In [ ]:
# Stage 04B: offline preparation of pinned dependencies, then one fixed GPU trial.
from pathlib import Path
import os,sys,json,hashlib,subprocess,time
TRIAL_BUDGET_STARTED=time.perf_counter()
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
inputs=Path('/kaggle/input')
resource_manifests=list(inputs.rglob('resources_manifest.json'))
assert len(resource_manifests)==1, 'Attach exactly one Stage 04B Resources notebook output'
RESOURCE_DIR=resource_manifests[0].parent
manifest=json.loads(resource_manifests[0].read_text())
for name,expected in manifest['wheel_files_sha256'].items():
    assert hashlib.sha256((RESOURCE_DIR/'wheels'/name).read_bytes()).hexdigest()==expected
subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps',
    '--find-links',str(RESOURCE_DIR/'wheels'),'transformers==4.56.2',
    'huggingface-hub==0.36.2','tokenizers==0.22.2','sentencepiece==0.2.1','protobuf==6.31.1'],check=True)
# A fresh Save & Run All process has not previously imported these packages.
assert 'transformers' not in sys.modules and 'tokenizers' not in sys.modules
RUNNER_SOURCE='"""Stage 04B: one fixed offline DeBERTa trial, with fresh weights per prompt fold."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport argparse\nimport gc\nimport hashlib\nimport importlib.metadata\nimport itertools\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport time\nimport traceback\nimport warnings\n\nos.environ[\'HF_HUB_OFFLINE\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\nos.environ[\'USE_TF\'] = \'0\'\nos.environ[\'USE_FLAX\'] = \'0\'\nos.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = \'0\'\nos.environ[\'CUBLAS_WORKSPACE_CONFIG\'] = \':4096:8\'\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom sklearn.metrics import balanced_accuracy_score, confusion_matrix\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\nfrom transformers import get_linear_schedule_with_warmup\n\nSEED = 20261016\nSOURCE_SHA = \'40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\'\nREFERENCE_SHA = \'15ca5d9d95fead2690a6c81559a3862f12143f83ad65684d1c584a2827e92ab3\'\nMODEL_ID = \'microsoft/deberta-v3-base\'\nREVISION = \'8ccc9b6f36199bec6961081d44eb72fb3f7353f3\'\nPLAN_COMMIT = \'8ab67d337b139059669455d605cf802eda9d4301\'\nVARIANT = \'deberta_v3_base_final_epoch\'\nREFERENCE = \'character_plus_style\'\nMAX_LENGTH, BATCH_SIZE, ACCUMULATION, EPOCHS = 384, 4, 4, 3\nMAX_SECONDS = 3 * 60 * 60\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef file_sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for block in iter(lambda: handle.read(1024*1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef save_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False))\n\n\ndef reset_seed():\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n\ndef check_budget(started):\n    # Reserve the last minute for saving status and any partial checkpoint.\n    if time.perf_counter() - started >= MAX_SECONDS - 60:\n        raise TimeoutError(\'Three-hour GPU budget reached; reserving time to preserve evidence\')\n\n\nclass TokenRows(Dataset):\n    def __init__(self, encoded, labels, indices):\n        self.encoded, self.labels = encoded, labels\n        self.indices = np.asarray(indices)\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, position):\n        index = int(self.indices[position])\n        row = {key:value[index] for key,value in self.encoded.items()}\n        row[\'labels\'] = int(self.labels[index])\n        return row\n\n\ndef make_collator(tokenizer):\n    def collate(rows):\n        labels = torch.tensor([r[\'labels\'] for r in rows], dtype=torch.long)\n        features = [{k:v for k,v in row.items() if k != \'labels\'} for row in rows]\n        batch = tokenizer.pad(features, padding=True, pad_to_multiple_of=8, return_tensors=\'pt\')\n        batch[\'labels\'] = labels\n        return batch\n    return collate\n\n\ndef weighted_loss_sum(logits, labels, weights):\n    # Explicit sum allows exact per-example normalization over partial accumulation groups.\n    return F.cross_entropy(logits.float(), labels, weight=weights, reduction=\'sum\')\n\n\ndef parameter_groups(model):\n    decay, no_decay = [], []\n    for name, value in model.named_parameters():\n        if not value.requires_grad:\n            continue\n        if name.endswith(\'bias\') or \'LayerNorm\' in name or \'layer_norm\' in name:\n            no_decay.append(value)\n        else:\n            decay.append(value)\n    assert len(decay) + len(no_decay) == sum(p.requires_grad for p in model.parameters())\n    return [{\'params\':decay, \'weight_decay\':0.01}, {\'params\':no_decay, \'weight_decay\':0.0}]\n\n\ndef environment_record():\n    packages = [\'torch\',\'transformers\',\'tokenizers\',\'sentencepiece\',\'protobuf\',\n                \'numpy\',\'pandas\',\'scikit-learn\',\'huggingface-hub\',\'safetensors\']\n    versions = {p:importlib.metadata.version(p) for p in packages}\n    return {\'python\':platform.python_version(), \'packages\':versions,\n            \'cuda\':torch.version.cuda, \'gpu\':torch.cuda.get_device_name(0),\n            \'visible_gpu_count\':torch.cuda.device_count(),\n            \'determinism\':\'Seeded, deterministic algorithms warn-only; GPU nondeterminism remains possible.\'}\n\n\ndef make_model(model_dir, labels):\n    model = AutoModelForSequenceClassification.from_pretrained(\n        model_dir, local_files_only=True, num_labels=len(labels),\n        id2label={i:label for i,label in enumerate(labels)},\n        label2id={label:i for i,label in enumerate(labels)})\n    return model.float().to(\'cuda:0\')\n\n\ndef paired_interval(oof, labels):\n    prompts = sorted(oof.prompt_id.unique())\n    gi = oof.prompt_id.map({x:i for i,x in enumerate(prompts)}).to_numpy()\n    yi = oof.model.map({x:i for i,x in enumerate(labels)}).to_numpy()\n    total = np.zeros((len(prompts),len(labels)),dtype=np.int64)\n    np.add.at(total,(gi,yi),1)\n    weights = np.random.default_rng(SEED).multinomial(len(prompts),\n        np.full(len(prompts),1/len(prompts)),size=2000)\n    denominator = weights @ total\n    assert (denominator > 0).all()\n    samples = {}\n    for variant in [VARIANT,REFERENCE]:\n        correct = np.zeros_like(total)\n        np.add.at(correct,(gi,yi),(oof[variant].to_numpy()==oof.model.to_numpy()).astype(int))\n        samples[variant] = ((weights @ correct)/denominator).mean(axis=1)\n    low,high = np.quantile(samples[VARIANT]-samples[REFERENCE],[.025,.975])\n    return {\'candidate\':VARIANT,\'reference\':REFERENCE,\n        \'delta_balanced_accuracy\':float(balanced_accuracy_score(oof.model,oof[VARIANT])-\n                                        balanced_accuracy_score(oof.model,oof[REFERENCE])),\n        \'lower_95\':float(low),\'upper_95\':float(high),\'unit\':\'prompt_id\',\'resamples\':2000,\'seed\':SEED,\n        \'limitation\':\'Conditional development predictions; no refitting/selection uncertainty or multiplicity correction.\'}\n\n\ndef run(data_dir, resource_dir, out_dir, budget_started=None):\n    started = budget_started if budget_started is not None else time.perf_counter()\n    if out_dir.exists():\n        raise FileExistsError(f\'Preserve the previous attempt before rerunning: {out_dir}\')\n    out_dir.mkdir(parents=True)\n    state = {\'stage\':\'04B_supervised_encoder\',\'status\':\'preflight\',\'plan_commit\':PLAN_COMMIT,\n             \'started_utc\':utc_now(),\'completed_folds\':[], \'gpu_budget_seconds\':MAX_SECONDS}\n    status_path = out_dir/\'run_status.json\'\n    save_json(status_path,state)\n    training_log, fold_rows, warning_log = [], [], []\n    prior_showwarning = warnings.showwarning\n    def record_warning(message, category, filename, lineno, file=None, line=None):\n        warning_log.append({\'category\':category.__name__,\'message\':str(message),\n                            \'filename\':filename,\'line\':lineno})\n        save_json(out_dir/\'warnings.json\',warning_log)\n        prior_showwarning(message,category,filename,lineno,file=file,line=line)\n    warnings.showwarning = record_warning\n    model, optimizer, scaler, scheduler = None, None, None, None\n    current_fold = None\n    try:\n        assert torch.cuda.is_available(), \'This recipe requires the declared included T4 GPU; no CPU fallback\'\n        torch.cuda.set_device(0)\n        assert \'T4\' in torch.cuda.get_device_name(0), \'Record an amendment before using different hardware\'\n        env = environment_record()\n        assert env[\'packages\'][\'transformers\']==\'4.56.2\'\n        assert env[\'packages\'][\'sentencepiece\']==\'0.2.1\'\n        assert tuple(int(x) for x in torch.__version__.split(\'+\')[0].split(\'.\')[:2]) >= (2,6)\n        save_json(out_dir/\'environment.json\',env)\n        assert file_sha(data_dir/\'train.csv\')==SOURCE_SHA\n        assert file_sha(resource_dir/\'stage03_oof_predictions.csv\')==REFERENCE_SHA\n        resource_manifest = json.loads((resource_dir/\'resources_manifest.json\').read_text())\n        assert resource_manifest[\'model_id\']==MODEL_ID and resource_manifest[\'revision\']==REVISION\n        model_dir = resource_dir/\'deberta-v3-base\'\n        for name, expected in resource_manifest[\'model_files_sha256\'].items():\n            assert file_sha(model_dir/name)==expected, f\'Checkpoint resource changed: {name}\'\n        save_json(out_dir/\'resources_manifest.json\',resource_manifest)\n        train = pd.read_csv(data_dir/\'train.csv\',dtype=str)\n        old = pd.read_csv(resource_dir/\'stage03_oof_predictions.csv\',\n                          dtype={\'id\':str,\'prompt_id\':str,\'model\':str})\n        assert train[[\'id\',\'prompt_id\',\'model\']].equals(old[[\'id\',\'prompt_id\',\'model\']])\n        assert len(train)==2699 and train.id.is_unique and train.prompt_id.nunique()==150\n        assert not train[[\'id\',\'prompt_id\',\'model\',\'text\']].isna().any().any()\n        assert set(old.fold)=={1,2,3,4,5} and old.groupby(\'prompt_id\').fold.nunique().eq(1).all()\n        labels = sorted(train.model.unique())\n        assert len(labels)==18\n        y = train.model.map({label:i for i,label in enumerate(labels)}).to_numpy()\n        assert np.isclose(balanced_accuracy_score(train.model,old[REFERENCE]),.35413124533929907)\n        save_json(out_dir/\'label_order.json\',labels)\n        tokenizer = AutoTokenizer.from_pretrained(model_dir,local_files_only=True,use_fast=False)\n        tokenizer.truncation_side = \'right\'\n        full_lengths = np.array([len(x) for x in tokenizer(train.text.tolist(),truncation=False,\n                                                         padding=False)[\'input_ids\']])\n        encoded = tokenizer(train.text.tolist(),truncation=True,max_length=MAX_LENGTH,padding=False)\n        assert max(len(x) for x in encoded[\'input_ids\'])<=MAX_LENGTH\n        save_json(out_dir/\'tokenization.json\',{\'max_length\':MAX_LENGTH,\'use_fast\':False,\n            \'pad_to_multiple_of\':8,\'truncated_rows\':int((full_lengths>MAX_LENGTH).sum()),\n            \'full_token_length_quantiles\':{str(q):float(np.quantile(full_lengths,q)) for q in [0,.5,.9,.95,1]}})\n        # Validate the real offline graph without fitting or evaluating held-out labels.\n        reset_seed()\n        model = make_model(model_dir,labels).eval()\n        with torch.inference_mode(),torch.autocast(device_type=\'cuda\',dtype=torch.float16):\n            probe = tokenizer([\'Offline model load check.\',\'A second short input.\'],padding=True,\n                               pad_to_multiple_of=8,return_tensors=\'pt\')\n            logits = model(**{k:v.to(\'cuda:0\') for k,v in probe.items()}).logits\n        assert logits.shape==(2,18) and torch.isfinite(logits).all()\n        del model, logits, probe\n        model = None\n        gc.collect()\n        torch.cuda.empty_cache()\n        state[\'status\']=\'running\'\n        state[\'preflight_passed_utc\']=utc_now()\n        save_json(status_path,state)\n        oof = old[[\'id\',\'prompt_id\',\'model\',\'fold\',REFERENCE]].copy()\n        oof[VARIANT] = None\n        all_logits = np.full((len(train),18),np.nan,dtype=np.float32)\n        print(\'PREFLIGHT PASSED:\',json.dumps(env),flush=True)\n        for fold in range(1,6):\n            current_fold = fold\n            check_budget(started)\n            reset_seed()\n            model = make_model(model_dir,labels)\n            valid = old.fold.eq(fold).to_numpy()\n            fit = ~valid\n            assert set(train.loc[fit,\'prompt_id\']).isdisjoint(train.loc[valid,\'prompt_id\'])\n            assert set(train.loc[fit,\'model\'])==set(labels)==set(train.loc[valid,\'model\'])\n            support = np.bincount(y[fit],minlength=18)\n            class_weights = 1.0/support\n            class_weights /= class_weights.mean()\n            loss_weights = torch.tensor(class_weights,dtype=torch.float32,device=\'cuda:0\')\n            generator = torch.Generator().manual_seed(SEED)\n            loader = DataLoader(TokenRows(encoded,y,np.flatnonzero(fit)),batch_size=BATCH_SIZE,\n                shuffle=True,generator=generator,num_workers=0,collate_fn=make_collator(tokenizer))\n            total_steps = EPOCHS * math.ceil(len(loader)/ACCUMULATION)\n            warmup_steps = math.ceil(total_steps*.10)\n            optimizer = torch.optim.AdamW(parameter_groups(model),lr=2e-5,betas=(.9,.999),eps=1e-8)\n            scheduler = get_linear_schedule_with_warmup(optimizer,warmup_steps,total_steps)\n            scaler = torch.amp.GradScaler(\'cuda\',init_scale=1024.0,growth_interval=2000)\n            fold_started = time.perf_counter()\n            update_count, overflow_count = 0, 0\n            for epoch in range(1,EPOCHS+1):\n                model.train()\n                epoch_loss_sum, epoch_rows = 0.0, 0\n                epoch_started = time.perf_counter()\n                iterator = iter(loader)\n                while True:\n                    group = list(itertools.islice(iterator,ACCUMULATION))\n                    if not group:\n                        break\n                    check_budget(started)\n                    group_rows = sum(len(batch[\'labels\']) for batch in group)\n                    optimizer.zero_grad(set_to_none=True)\n                    for batch in group:\n                        targets = batch.pop(\'labels\').to(\'cuda:0\')\n                        inputs = {k:v.to(\'cuda:0\') for k,v in batch.items()}\n                        with torch.autocast(device_type=\'cuda\',dtype=torch.float16):\n                            logits = model(**inputs).logits\n                            loss_sum = weighted_loss_sum(logits,targets,loss_weights)\n                            loss = loss_sum/group_rows\n                        if not torch.isfinite(loss):\n                            raise FloatingPointError(\'Nonfinite training loss\')\n                        scaler.scale(loss).backward()\n                        epoch_loss_sum += float(loss_sum.detach().cpu())\n                        epoch_rows += len(targets)\n                    scaler.unscale_(optimizer)\n                    norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)\n                    previous_scale = scaler.get_scale()\n                    scaler.step(optimizer)\n                    scaler.update()\n                    if scaler.get_scale()<previous_scale or not torch.isfinite(norm):\n                        overflow_count += 1\n                        save_json(out_dir/f\'fold_{fold}_overflow.json\',{\'fold\':fold,\'epoch\':epoch,\n                            \'completed_optimizer_steps\':update_count,\'overflow_count\':overflow_count,\n                            \'old_scale\':previous_scale,\'new_scale\':scaler.get_scale()})\n                        raise FloatingPointError(\'Nonfinite gradients/AMP overflow; trial retained as incomplete\')\n                    scheduler.step()\n                    update_count += 1\n                    if update_count%25==0:\n                        print(f\'fold={fold} epoch={epoch} update={update_count}/{total_steps} \'\n                              f\'elapsed={time.perf_counter()-started:.1f}s\',flush=True)\n                assert epoch_rows==int(fit.sum())\n                record = {\'fold\':fold,\'epoch\':epoch,\'rows\':epoch_rows,\n                    \'mean_weighted_training_loss\':epoch_loss_sum/epoch_rows,\n                    \'cumulative_optimizer_steps\':update_count,\'amp_overflow_steps\':overflow_count,\n                    \'seconds\':time.perf_counter()-epoch_started,\'validation_performed\':False}\n                training_log.append(record)\n                save_json(out_dir/\'training_log.json\',training_log)\n                print(\'EPOCH:\',json.dumps(record),flush=True)\n            assert update_count==total_steps\n            for parameter in model.parameters():\n                if not torch.isfinite(parameter).all():\n                    raise FloatingPointError(\'Nonfinite final model parameter\')\n            model.eval()\n            valid_loader = DataLoader(TokenRows(encoded,y,np.flatnonzero(valid)),batch_size=BATCH_SIZE,\n                shuffle=False,num_workers=0,collate_fn=make_collator(tokenizer))\n            scores = []\n            for batch in valid_loader:\n                check_budget(started)\n                batch.pop(\'labels\')\n                with torch.inference_mode(),torch.autocast(device_type=\'cuda\',dtype=torch.float16):\n                    logits = model(**{k:v.to(\'cuda:0\') for k,v in batch.items()}).logits\n                scores.append(logits.float().cpu().numpy())\n            scores = np.concatenate(scores)\n            assert scores.shape==(int(valid.sum()),18) and np.isfinite(scores).all()\n            all_logits[valid] = scores\n            predictions = np.asarray(labels)[scores.argmax(axis=1)]\n            oof.loc[valid,VARIANT] = predictions\n            score = float(balanced_accuracy_score(train.loc[valid,\'model\'],predictions))\n            checkpoint_dir = out_dir/f\'fold_{fold}_checkpoint\'\n            model.save_pretrained(checkpoint_dir,safe_serialization=True)\n            tokenizer.save_pretrained(checkpoint_dir)\n            checkpoint_hashes = {p.name:file_sha(p) for p in sorted(checkpoint_dir.glob(\'*\')) if p.is_file()}\n            fold_rows.append({\'variant\':VARIANT,\'fold\':fold,\'balanced_accuracy\':score,\n                \'rows\':int(valid.sum()),\'prompts\':int(train.loc[valid,\'prompt_id\'].nunique()),\n                \'seconds\':time.perf_counter()-fold_started,\'epochs\':EPOCHS,\'optimizer_steps\':update_count,\n                \'amp_overflow_steps\':overflow_count,\'warmup_steps\':warmup_steps,\n                \'training_class_support\':support.tolist(),\'training_class_weights\':class_weights.tolist(),\n                \'checkpoint_sha256\':checkpoint_hashes})\n            save_json(out_dir/\'fold_records.json\',fold_rows)\n            oof.to_csv(out_dir/\'oof_predictions.csv\',index=False)\n            np.save(out_dir/\'oof_logits.npy\',all_logits,allow_pickle=False)\n            state[\'completed_folds\'].append(fold)\n            save_json(status_path,state)\n            print(f\'FINAL_EPOCH_FOLD_SCORE fold={fold} balanced_accuracy={score:.9f}\',flush=True)\n            loss,loss_sum,inputs,targets,norm,parameter = None,None,None,None,None,None\n            del model,optimizer,scheduler,scaler,loss_weights,loader,valid_loader\n            model,optimizer,scheduler,scaler = None,None,None,None\n            gc.collect()\n            torch.cuda.empty_cache()\n        assert oof[VARIANT].notna().all() and np.isfinite(all_logits).all()\n        variants = [VARIANT,REFERENCE]\n        comparisons,recall_rows,length_rows,fold_table = [],[],[],[]\n        bins = pd.cut(train.text.str.split().str.len(),[-np.inf,80,150,np.inf],\n                      labels=[\'<=80 words\',\'81-150 words\',\'>150 words\'])\n        for variant in variants:\n            score = float(balanced_accuracy_score(oof.model,oof[variant]))\n            cm = confusion_matrix(oof.model,oof[variant],labels=labels)\n            recalls = cm.diagonal()/cm.sum(axis=1)\n            assert np.isclose(score,recalls.mean(),atol=1e-12)\n            comparisons.append({\'variant\':variant,\'balanced_accuracy\':score})\n            recall_rows.extend({\'variant\':variant,\'model\':label,\'recall\':float(value),\'support\':int(support)}\n                               for label,value,support in zip(labels,recalls,cm.sum(axis=1)))\n            for fold in range(1,6):\n                selected = oof.fold.eq(fold)\n                fold_table.append({\'variant\':variant,\'fold\':fold,\n                    \'balanced_accuracy\':float(balanced_accuracy_score(oof.loc[selected,\'model\'],oof.loc[selected,variant]))})\n            for name in bins.cat.categories:\n                selected = bins.eq(name)\n                length_rows.append({\'variant\':variant,\'word_count_bin\':name,\'rows\':int(selected.sum()),\n                    \'classes_present\':int(train.loc[selected,\'model\'].nunique()),\n                    \'balanced_accuracy_present_classes\':float(balanced_accuracy_score(oof.loc[selected,\'model\'],oof.loc[selected,variant]))})\n        interval = paired_interval(oof,labels)\n        pd.DataFrame(comparisons).to_csv(out_dir/\'comparison.csv\',index=False)\n        pd.DataFrame(fold_table).to_csv(out_dir/\'fold_scores.csv\',index=False)\n        pd.DataFrame(recall_rows).to_csv(out_dir/\'per_model_recall.csv\',index=False)\n        pd.DataFrame(length_rows).to_csv(out_dir/\'length_diagnostics.csv\',index=False)\n        for i,label in enumerate(labels):\n            oof[f\'logit_{i:02d}\'] = all_logits[:,i]\n        oof.to_csv(out_dir/\'oof_predictions.csv\',index=False,float_format=\'%.9g\')\n        result = {**state,\'status\':\'complete\',\'finished_utc\':utc_now(),\n            \'source_sha256\':SOURCE_SHA,\'reference_sha256\':REFERENCE_SHA,\n            \'model_id\':MODEL_ID,\'revision\':REVISION,\'comparisons\':comparisons,\n            \'paired_difference\':interval,\'fold_records\':fold_rows,\'warnings\':warning_log,\n            \'environment\':env,\'total_seconds\':time.perf_counter()-started,\n            \'selection\':\'Final epoch 3, fixed prospectively; no validation-driven epoch or parameter selection.\',\n            \'official_submission_made\':False,\'private_test_result\':None}\n        save_json(out_dir/\'metrics.json\',result)\n        state.update(status=\'complete\',finished_utc=result[\'finished_utc\'])\n        save_json(status_path,state)\n        print(\'TRIAL_COMPLETE_JSON_BEGIN\',flush=True)\n        print(json.dumps(result,indent=2),flush=True)\n        print(\'TRIAL_COMPLETE_JSON_END\',flush=True)\n        print(pd.DataFrame(recall_rows).to_string(index=False),flush=True)\n        return result\n    except Exception as error:\n        state.update(status=\'incomplete\',failed_utc=utc_now(),error_type=type(error).__name__,\n                     error=str(error),current_fold=current_fold)\n        save_json(status_path,state)\n        (out_dir/\'failure_traceback.txt\').write_text(traceback.format_exc())\n        save_json(out_dir/\'training_log.json\',training_log)\n        if model is not None:\n            try:\n                model.save_pretrained(out_dir/f\'incomplete_fold_{current_fold}\',safe_serialization=True)\n            except Exception as checkpoint_error:\n                (out_dir/\'checkpoint_save_error.txt\').write_text(str(checkpoint_error))\n        print(\'TRIAL_INCOMPLETE:\',json.dumps(state),flush=True)\n        raise\n    finally:\n        warnings.showwarning = prior_showwarning\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--data-dir\',type=Path,required=True)\n    parser.add_argument(\'--resource-dir\',type=Path,required=True)\n    parser.add_argument(\'--out-dir\',type=Path,required=True)\n    args = parser.parse_args()\n    run(args.data_dir,args.resource_dir,args.out_dir)\n'
assert hashlib.sha256(RUNNER_SOURCE.encode()).hexdigest()=='dfff975f59870518bdaedd614de48aa7dd4e9c4fd9ac9b10b4e433b35d440b9a'
Path('/kaggle/working/supervised_encoder_trial.py').write_text(RUNNER_SOURCE)
namespace={'__name__':'stage04b_runner','__file__':'/kaggle/working/supervised_encoder_trial.py'}
exec(compile(RUNNER_SOURCE,namespace['__file__'],'exec'),namespace)
DATA_DIR=next(p for p in [Path('/kaggle/input/competitions/name-that-llm'),Path('/kaggle/input/name-that-llm')]
              if (p/'train.csv').is_file())
result=namespace['run'](DATA_DIR,RESOURCE_DIR,Path('/kaggle/working/stage04b'),TRIAL_BUDGET_STARTED)
